In [ ]:
import pandas as pd
import requests
import os
import time
from pathlib import Path
import logging
from typing import Set, Tuple, List, Dict, Any, Optional
import sys
import random
import concurrent.futures
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('wowy_scraper.log'),
        logging.StreamHandler()
    ]
)

# Create a session with retry capabilities
def create_session_with_retries(retries=5, backoff_factor=0.5, 
                               status_forcelist=(500, 502, 503, 504, 429)):
    """Create a requests Session with automatic retries."""
    session = requests.Session()
    retry_strategy = Retry(
        total=retries,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
        allowed_methods=["GET"]
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

# Use a session for all requests
session = create_session_with_retries()

# Component parts for generating dynamic headers
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/114.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.4 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 16_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.5 Mobile/15E148 Safari/604.1",
    "Mozilla/5.0 (iPad; CPU OS 16_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.5 Mobile/15E148 Safari/604.1",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Edge/125.0.0.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/115.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
]

ACCEPT_TYPES = [
    "application/json, text/plain, */*",
    "application/json",
    "application/json, text/javascript, */*; q=0.01",
    "*/*",
    "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
]

ACCEPT_LANGUAGES = [
    "en-US,en;q=0.9",
    "en-US,en;q=0.5",
    "en-GB,en;q=0.7,en-US;q=0.3",
    "en-US,en;q=0.8,es;q=0.5",
    "en-CA,en;q=0.9,fr-CA;q=0.7"
]

ACCEPT_ENCODINGS = [
    "gzip, deflate, br",
    "gzip, deflate",
    "br;q=1.0, gzip;q=0.8, *;q=0.1"
]

REFERERS = [
    "https://stats.nba.com/",
    "https://www.nba.com/",
    "https://www.nba.com/stats/",
    "https://www.nba.com/players/",
    "https://www.nba.com/teams/",
    "https://www.espn.com/nba/",
    "https://www.basketball-reference.com/"
]

HOSTS = [
    "stats.nba.com",
    "api.nba.com",
    "data.nba.com",
    "www.nba.com"
]

CONNECTIONS = [
    "keep-alive",
    "close"
]

CACHE_CONTROLS = [
    "max-age=0",
    "no-cache",
    "max-age=300"
]

def generate_random_headers():
    """Generate random realistic headers to avoid detection."""
    user_agent = random.choice(USER_AGENTS)
    
    # Build header with mandatory fields
    headers = {
        "User-Agent": user_agent,
        "Accept": random.choice(ACCEPT_TYPES),
    }
    
    # Add optional headers with some randomness
    if random.random() > 0.2:  # 80% chance to include
        headers["Accept-Language"] = random.choice(ACCEPT_LANGUAGES)
    
    if random.random() > 0.2:  # 80% chance to include
        headers["Accept-Encoding"] = random.choice(ACCEPT_ENCODINGS)
        
    if random.random() > 0.1:  # 90% chance to include
        headers["Referer"] = random.choice(REFERERS)
        
    if random.random() > 0.3:  # 70% chance to include
        headers["Host"] = random.choice(HOSTS)
        
    if random.random() > 0.3:  # 70% chance to include
        headers["Connection"] = random.choice(CONNECTIONS)
        
    # Add some extra headers occasionally
    if random.random() > 0.7:  # 30% chance to include
        headers["Cache-Control"] = random.choice(CACHE_CONTROLS)
        
    if random.random() > 0.8:  # 20% chance to include
        headers["Pragma"] = "no-cache"
        
    if random.random() > 0.8:  # 20% chance to include
        headers["DNT"] = "1"
        
    if random.random() > 0.9:  # 10% chance to include
        headers["Upgrade-Insecure-Requests"] = "1"
        
    # Add some random cookies occasionally
    if random.random() > 0.9:  # 10% chance to include
        cookie_id = f"{random.randint(10000000, 99999999)}"
        session_id = f"session_{random.randint(1000000, 9999999)}"
        headers["Cookie"] = f"_ga=GA1.2.{cookie_id}.{int(time.time() - random.randint(1000000, 9999999))}; _gid=GA1.2.{cookie_id}; sessionid={session_id}"
        
    return headers

# Alias for backwards compatibility
get_random_headers = generate_random_headers

def wowy_shift(team_id: str, player1_id: str, seasons: List[str], ps=False, max_retries=5) -> pd.DataFrame:
    """
    Get WOWY (With Or Without You) stats for a player from the NBA stats API.
    
    Args:
        team_id: NBA team ID
        player1_id: NBA player ID
        seasons: List of seasons (e.g., ["2021-22"])
        ps: False for regular season, True for playoffs, 'all' for both
        max_retries: Maximum number of retries for failed requests
        
    Returns:
        DataFrame with WOWY stats
    """
    if ps == False:
        s_type = 'Regular Season'
    elif ps == 'all':
        s_type = 'All'
    else:
        s_type = 'Playoffs'
    
    seasons_str = ",".join(seasons)
    
    # Data for both requests
    datasets = [
        {
            "param_key": "0Exactly1OnFloor",
            "label": True,  # Player on
        },
        {
            "param_key": "0Exactly0OnFloor",
            "label": False,  # Player off
        }
    ]
    
    combined_data = []
    
    # Create a dedicated session for this request with its own retry strategy
    request_session = create_session_with_retries(retries=max_retries)
    
    for dataset in datasets:
        retry_count = 0
        success = False
        
        # Track headers that have failed for this request
        failed_headers = set()
        
        while not success and retry_count < max_retries:
            try:
                # Generate new headers for each retry
                headers = generate_random_headers()
                
                # Construct a fingerprint of the headers to avoid reusing failed ones
                headers_hash = hash(frozenset(headers.items()))
                if headers_hash in failed_headers and len(failed_headers) < 10:
                    # If we've tried these headers before and they failed, generate new ones
                    continue
                
                wowy_params = {
                    dataset["param_key"]: player1_id,
                    "TeamId": team_id,
                    "Season": seasons_str,
                    "SeasonType": s_type,
                    "Type": "Player",
                }
                
                # Add some jitter to request parameters to appear more human-like
                if random.random() > 0.7:  # 30% chance
                    # Add a cache buster
                    wowy_params["_"] = int(time.time() * 1000)
                
                wowy_url = "https://api.pbpstats.com/get-wowy-stats/nba"
                
                # Set custom timeout with jitter
                timeout = 10 + random.uniform(0, 5)
                
                response = request_session.get(
                    wowy_url, 
                    params=wowy_params, 
                    headers=headers, 
                    timeout=timeout
                )
                
                if response.status_code != 200:
                    failed_headers.add(headers_hash)
                    logging.warning(f"Got status code {response.status_code}, retrying {retry_count+1}/{max_retries}")
                    retry_count += 1
                    
                    # Dynamic backoff based on response status
                    if response.status_code == 429:  # Rate limit
                        wait_time = 5 + 2 ** retry_count + random.uniform(1, 5)
                    else:
                        wait_time = 1 + retry_count + random.uniform(0, 2)
                        
                    logging.info(f"Waiting {wait_time:.2f}s before retry")
                    time.sleep(wait_time)
                    continue
                
                # Parse response and check for valid data
                try:
                    wowy = response.json()
                except Exception as json_e:
                    failed_headers.add(headers_hash)
                    logging.warning(f"Failed to parse JSON: {json_e}, retrying {retry_count+1}/{max_retries}")
                    retry_count += 1
                    time.sleep(1 + random.uniform(0, 2))
                    continue
                
                if not wowy.get("multi_row_table_data"):
                    failed_headers.add(headers_hash)
                    logging.warning(f"No data returned for {player1_id} with {team_id}, retrying {retry_count+1}/{max_retries}")
                    retry_count += 1
                    time.sleep(1 + random.uniform(0, 2))
                    continue
                
                # Process successful response
                player_stats = wowy["multi_row_table_data"]
                df = pd.DataFrame(player_stats)
                df['on'] = dataset["label"]
                combined_data.append(df)
                success = True
                
                # Add slight delay between successful requests with jitter
                delay = 0.3 + random.uniform(0, 0.7)
                logging.debug(f"Request successful, waiting {delay:.2f}s before next request")
                time.sleep(delay)
                
            except requests.exceptions.Timeout:
                failed_headers.add(headers_hash)
                retry_count += 1
                wait_time = 2 ** retry_count  # Exponential backoff
                logging.warning(f"Timeout for {player1_id} with {team_id}. Retrying in {wait_time}s")
                time.sleep(wait_time)
                
            except requests.exceptions.ConnectionError:
                failed_headers.add(headers_hash)
                retry_count += 1
                wait_time = 5 + 2 ** retry_count  # More aggressive backoff for connection issues
                logging.warning(f"Connection error for {player1_id} with {team_id}. Retrying in {wait_time}s")
                time.sleep(wait_time)
                
            except Exception as e:
                failed_headers.add(headers_hash)
                retry_count += 1
                wait_time = 2 ** retry_count  # Exponential backoff
                logging.error(f"Error processing {player1_id} with {team_id}: {e}. Retrying in {wait_time}s")
                time.sleep(wait_time)
        
        if not success:
            logging.error(f"Failed to get data for {player1_id} with {team_id} after {max_retries} retries")
            # Return empty DataFrame if one request succeeded but not the other
            if combined_data:
                return combined_data[0]
            # Return completely empty DataFrame if both failed
            return pd.DataFrame()
    
    # If we get here, both requests were successful
    return pd.concat(combined_data) if combined_data else pd.DataFrame()

def setup_folders(base_year: int, end_year: int, ps=False) -> None:
    """Create folders for each season if they don't exist."""
    trail = 'ps' if ps else ''
    for year in range(base_year, end_year + 1):
        Path(f"data/{year}{trail}").mkdir(parents=True, exist_ok=True)

def get_processed_combinations(year: int, ps=False) -> Set[Tuple[str, str]]:
    """Get already processed player-team combinations for a given year."""
    trail = 'ps' if ps else ''
    year_dir = Path(f"data/{year}{trail}")
    processed = set()
    
    if year_dir.exists():
        for file in year_dir.glob("*.csv"):
            nba_id = file.stem
            try:
                df = pd.read_csv(file)
                team_ids = df['TeamId'].unique()
                for team_id in team_ids:
                    processed.add((nba_id, str(team_id)))
            except Exception as e:
                logging.error(f"Error reading file {file}: {e}")
    
    return processed

def save_data_chunk(output_file: Path, data: pd.DataFrame) -> None:
    """Save data to CSV, handling existing files."""
    try:
        if output_file.exists():
            existing_data = pd.read_csv(output_file)
            combined_data = pd.concat([existing_data, data], ignore_index=True)
            combined_data.drop_duplicates().to_csv(output_file, index=False)
        else:
            data.to_csv(output_file, index=False)
        logging.info(f"Successfully saved data to {output_file}")
    except Exception as e:
        logging.error(f"Error saving data to {output_file}: {e}")
        # Create backup in case of error
        backup_file = output_file.with_suffix('.backup.csv')
        try:
            data.to_csv(backup_file, index=False)
            logging.info(f"Created backup file {backup_file}")
        except Exception as backup_e:
            logging.error(f"Failed to create backup: {backup_e}")

def process_player_team_combination(nba_id: str, team_id: str, year: int, 
                                   seasons: List[str], is_postseason: bool) -> Optional[pd.DataFrame]:
    """Process a single player-team combination."""
    trail = 'ps' if is_postseason else ''
    output_file = Path(f"data/{int(year)}{trail}/{int(nba_id)}.csv")
    
    try:
        logging.info(f"Processing {nba_id} - {team_id} for {year}")
        
        # Add jitter to avoid synchronized requests
        time.sleep(random.uniform(0.1, 0.5))
        
        result = wowy_shift(
            team_id=team_id,
            player1_id=str(int(nba_id)),
            seasons=seasons,
            ps=is_postseason
        )
        
        if result.empty:
            logging.warning(f"No data returned for {nba_id} - {team_id} for {year}")
            return None
        
        # Save data immediately after retrieval
        save_data_chunk(output_file, result)
        
        return result
        
    except Exception as e:
        logging.error(f"Error processing {nba_id} - {team_id} for {year}: {e}")
        return None

def chunk_player_list(player_teams: List[Tuple[str, str]], chunk_size=10) -> List[List[Tuple[str, str]]]:
    """Split player list into chunks for batch processing and saving."""
    return [player_teams[i:i+chunk_size] for i in range(0, len(player_teams), chunk_size)]

def process_season_data(year: int, is_postseason: bool, index_df: pd.DataFrame, 
                       processed_combinations: Set[Tuple[str, str]], concurrent_requests=5) -> None:
    """Process data for a single season with concurrent requests and chunked saving."""
    trail = 'ps' if is_postseason else ''
    season_start = str(year - 1)
    season_end = str(year)
    seasons = [f"{season_start}-{season_end[-2:]}"]
    
    index_df['nba_id'] = index_df['nba_id'].astype(int)
    
    # Get all unique player-team combinations for this season that haven't been processed
    player_teams = []
    for _, row in index_df[index_df['year'] == year].iterrows():
        nba_id, team_id = str(int(row['nba_id'])), str(row['team_id'])
        if (nba_id, team_id) not in processed_combinations:
            player_teams.append((nba_id, team_id))
    
    if not player_teams:
        logging.info(f"No new data to process for {year} {'playoffs' if is_postseason else 'regular season'}")
        return
    
    logging.info(f"Processing {len(player_teams)} player-team combinations for {year} {'playoffs' if is_postseason else 'regular season'}")
    
    # Shuffle the player_teams list to randomize the order of requests
    random.shuffle(player_teams)
    
    # Process in chunks with concurrent requests within each chunk
    chunks = chunk_player_list(player_teams, chunk_size=20)  # Process 20 players at a time (increased from 15)
    
    # Use adaptive concurrency with circuit breaker pattern
    current_concurrency = concurrent_requests
    max_concurrency = 8  # Maximum number of concurrent requests
    min_concurrency = 2  # Minimum number of concurrent requests
    failure_threshold = 0.3  # If more than 30% of requests fail, reduce concurrency
    success_streak = 0  # Track consecutive successful chunks
    failure_streak = 0  # Track consecutive failed chunks
    
    # Track global success/failure stats
    total_requests = 0
    failed_requests = 0
    
    for i, chunk in enumerate(chunks):
        logging.info(f"Processing chunk {i+1}/{len(chunks)} for {year} (concurrency: {current_concurrency})")
        
        # Track failures in this chunk
        chunk_failures = 0
        chunk_total = len(chunk)
        
        with concurrent.futures.ThreadPoolExecutor(max_workers=current_concurrency) as executor:
            futures = {
                executor.submit(
                    process_player_team_combination, 
                    nba_id, team_id, year, seasons, is_postseason
                ): (nba_id, team_id) 
                for nba_id, team_id in chunk
            }
            
            for future in concurrent.futures.as_completed(futures):
                nba_id, team_id = futures[future]
                total_requests += 1
                
                try:
                    result = future.result()
                    if result is not None:
                        # Add to processed set
                        processed_combinations.add((nba_id, team_id))
                    else:
                        # Count empty results as partial failures
                        chunk_failures += 0.5
                        failed_requests += 0.5
                except Exception as e:
                    chunk_failures += 1
                    failed_requests += 1
                    logging.error(f"Executor error for {nba_id} - {team_id}: {e}")
        
        # Update checkpoint more frequently (after each chunk)
        checkpoint_file = Path("scraper_checkpoint.txt")
        with open(checkpoint_file, 'w') as f:
            f.write(f"{year if is_postseason else 2010},{year if not is_postseason else 2010}")
            
        # Adaptive concurrency based on failure rate in this chunk
        chunk_failure_rate = chunk_failures / chunk_total if chunk_total > 0 else 0
        global_failure_rate = failed_requests / total_requests if total_requests > 0 else 0
        
        logging.info(f"Chunk {i+1} completed with {chunk_failure_rate:.1%} failure rate (global: {global_failure_rate:.1%})")
        
        # Adjust concurrency based on failure rates
        if chunk_failure_rate >= failure_threshold:
            failure_streak += 1
            success_streak = 0
            if current_concurrency > min_concurrency and failure_streak >= 2:
                # Reduce concurrency after 2 consecutive high-failure chunks
                current_concurrency = max(current_concurrency - 1, min_concurrency)
                failure_streak = 0  # Reset streak
                logging.info(f"Reducing concurrency to {current_concurrency} due to high failure rate")
        else:
            failure_streak = 0
            success_streak += 1
            if current_concurrency < max_concurrency and success_streak >= 3:
                # Increase concurrency after 3 consecutive successful chunks
                current_concurrency = min(current_concurrency + 1, max_concurrency)
                success_streak = 0  # Reset streak
                logging.info(f"Increasing concurrency to {current_concurrency} due to sustained success")
        
        # Dynamic wait between chunks based on failure rate and current concurrency
        if i < len(chunks) - 1:  # No need to wait after the last chunk
            wait_time = 0
            if chunk_failure_rate > 0.5:  # High failure rate
                wait_time = 10 + random.uniform(0, 5)  # Longer wait time
            elif chunk_failure_rate > 0.2:  # Moderate failure rate
                wait_time = 5 + random.uniform(0, 3)
            else:  # Low failure rate
                wait_time = 2 + random.uniform(0, 2)
                
            # Adjust wait time based on concurrency
            wait_time = wait_time * (current_concurrency / 3)
                
            logging.info(f"Waiting {wait_time:.2f}s before processing next chunk...")
            time.sleep(wait_time)

def load_index_data():
    """Load index data with retry logic."""
    max_retries = 5
    
    for attempt in range(max_retries):
        try:
            index_reg = pd.read_csv('https://raw.githubusercontent.com/gabriel1200/site_Data/refs/heads/master/index_master.csv')
            index_reg.dropna(subset=['nba_id', 'team_id'], inplace=True)
            index_reg = index_reg[index_reg.team != 'TOT']
            
            index_ps = pd.read_csv('https://raw.githubusercontent.com/gabriel1200/site_Data/refs/heads/master/index_master_ps.csv')
            index_ps = index_ps[index_ps.team != 'TOT']
            
            return index_reg, index_ps
            
        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                logging.error(f"Error loading index files (attempt {attempt+1}/{max_retries}): {e}. Retrying in {wait_time}s")
                time.sleep(wait_time)
            else:
                logging.error(f"Failed to load index files after {max_retries} attempts: {e}")
                raise

def validate_and_repair_data_files(year_range, is_postseason=False):
    """Validate existing data files and attempt repairs if they're corrupted."""
    trail = 'ps' if is_postseason else ''
    repaired_count = 0
    
    for year in year_range:
        year_dir = Path(f"data/{year}{trail}")
        if not year_dir.exists():
            continue
            
        for file in year_dir.glob("*.csv"):
            try:
                # Try reading the file
                df = pd.read_csv(file)
            except Exception as e:
                logging.warning(f"Corrupted file detected: {file} - {e}")
                
                # Check if a backup exists
                backup_file = file.with_suffix('.backup.csv')
                if backup_file.exists():
                    try:
                        # Try

if __name__ == "__main__":
    main()